# Framework vs Hand-Rolled [Security - Module 02, Notebook 08]

> **MLCourse - Agentic AI - Production Security**

You have now seen four ways to build a guardrail:

| Notebook | Approach | Mechanism |
|---|---|---|
| `01` | hand-rolled | regex, Pydantic, plain Python |
| `05` | NeMo Guardrails | Colang flows, embedding intent matching |
| `06` | Guardrails-AI | validators with explicit `on_fail` policies |
| `07` | Llama Guard | a fine-tuned safety classifier model |

The obvious lesson to draw is "frameworks are the professional choice, the
hand-rolled version in notebook `01` was just for teaching". **That lesson is
wrong**, and this notebook exists to say so clearly.

For a large fraction of real guardrails -- probably the majority -- the twenty
lines of Python you wrote in notebook `01` are the *better engineering choice*,
and adding a framework makes the system slower, heavier, and harder to debug
for no gain in safety.

This notebook measures the trade-offs rather than asserting them, and ends with
a decision procedure you can actually apply.

### What you will learn

1. Real latency numbers for each approach, measured here.
2. What each dependency actually costs you.
3. Why debuggability degrades as you move up the stack.
4. The cases where a framework is clearly right.
5. The cases where hand-rolled is clearly right -- and these are common.

### Key takeaway up front

> Choose the **cheapest mechanism that can express the rule**. Escalate only
> when the rule genuinely cannot be written down.

### 1. Setup

We import all four approaches side by side so we can measure them on equal
footing. Everything here runs locally except the optional Groq call.

### Setup


In [ ]:
import os
import re
import sys
import time
import json
import statistics
import warnings
import importlib.metadata as md
from pathlib import Path

import requests
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

def find_env(depth: int = 8):
    p = Path.cwd()
    for _ in range(depth):
        candidate = p / "03_agentic_ai" / ".env"
        if candidate.is_file():
            return candidate
        p = p.parent
    return None

ENV_PATH = find_env()
load_dotenv(ENV_PATH, override=False)

OLLAMA = "http://localhost:11434"

def ollama_up() -> bool:
    try:
        return requests.get(f"{OLLAMA}/api/tags", timeout=3).status_code == 200
    except Exception:
        return False

OLLAMA_UP = ollama_up()
GUARD_MODEL = None
if OLLAMA_UP:
    names = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags", timeout=5).json()["models"]]
    guards = [n for n in names if "guard" in n.lower()]
    GUARD_MODEL = guards[0] if guards else None

print("Module 02 / Notebook 08: Framework vs Hand-Rolled")
print(f"env file      : {ENV_PATH}")
print(f"ollama up     : {OLLAMA_UP}")
print(f"guard model   : {GUARD_MODEL}")
print(f"guardrails-ai : {md.version('guardrails-ai')}")
print(f"nemoguardrails: {md.version('nemoguardrails')}")


### 2. The same rule, three ways

To compare fairly we need one rule implemented in each style. We will use a
genuinely narrow, well-understood check -- exactly the kind that comes up
constantly in real systems:

> **The text must not contain an API key, an SSN, or a credit card number.**

This rule is fully specifiable. There is no ambiguity about what an SSN looks
like. Keep that property in mind; it is the crux of the whole notebook.

### Implementation A: hand-rolled (the notebook 01 style)


In [ ]:
SECRET_PATTERNS = [
    (re.compile(r"\bsk-[A-Za-z0-9]{8,}\b"), "API_KEY"),
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),  "SSN"),
    (re.compile(r"\b(?:\d{4}[ -]?){3}\d{4}\b"), "CARD"),
]

def handrolled_check(text: str):
    """Returns (ok, reason). ~10 lines. No dependencies. Fails closed."""
    for pattern, label in SECRET_PATTERNS:
        if pattern.search(text):
            return False, f"{label} detected"
    return True, ""

print("Implementation A: hand-rolled")
print(f"  lines of code : ~10")
print(f"  dependencies  : none (stdlib `re`)")
for sample in ["your order ships tuesday", "key is sk-abc123def456"]:
    print(f"  {handrolled_check(sample)}  <- {sample!r}")


### Implementation B: Guardrails-AI


In [ ]:
from guardrails import Guard, OnFailAction, settings
settings.disable_tracing = True
from guardrails.validators import Validator, PassResult, FailResult, register_validator

@register_validator(name="nb08-no-secrets", data_type="string")
class NoSecrets(Validator):
    def validate(self, value: str, metadata: dict):
        for pattern, label in SECRET_PATTERNS:
            if pattern.search(value):
                return FailResult(error_message=f"{label} detected")
        return PassResult()

gai_guard = Guard().use(NoSecrets(on_fail=OnFailAction.EXCEPTION))

def guardrails_ai_check(text: str):
    try:
        gai_guard.validate(text)
        return True, ""
    except Exception as e:
        return False, str(e)[:60]

print("Implementation B: Guardrails-AI")
print("  identical regex, wrapped in a Validator + Guard")
for sample in ["your order ships tuesday", "key is sk-abc123def456"]:
    print(f"  {guardrails_ai_check(sample)}  <- {sample!r}")


### Implementation C: Llama Guard classifier


In [ ]:
def llama_guard_check(text: str):
    """The learned classifier. Note: it was never told about our regex rule."""
    response = requests.post(
        f"{OLLAMA}/api/chat",
        json={
            "model": GUARD_MODEL,
            "messages": [{"role": "user", "content": text}],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=180,
    )
    raw = response.json()["message"]["content"].strip()
    lines = [ln for ln in raw.splitlines() if ln.strip()]
    ok = bool(lines) and lines[0].lower() == "safe"
    return ok, "" if ok else raw.replace("\n", "/")

if GUARD_MODEL:
    print("Implementation C: Llama Guard")
    for sample in ["your order ships tuesday", "key is sk-abc123def456"]:
        print(f"  {llama_guard_check(sample)}  <- {sample!r}")
else:
    raise RuntimeError("Llama Guard not installed; run `ollama pull llama-guard3:1b`")


### Already, a result worth pausing on

Look at what Llama Guard said about `"key is sk-abc123def456"`.

It very likely returned **safe**. That is not a malfunction. Llama Guard was
trained to detect *hazardous content* -- violence, weapons, hate, self-harm.
A leaked API key is not in its taxonomy. It is not a safety hazard in the sense
the model was trained on; it is a business and security problem specific to you.

**The most sophisticated tool in the module fails at this task, and ten lines of
regex succeed perfectly.** Sophistication is not generality. A learned model is
only better on the distribution it learned.

### 3. Measuring latency

Now the numbers. We run each implementation many times over the same inputs and
report the median per-call cost.

### Benchmark


In [ ]:
SAMPLES = [
    "Your order ships on Tuesday and should arrive by Friday.",
    "Please confirm the delivery address for order 44812.",
    "Here is the key you wanted: sk-abc123def456xyz",
    "My social is 123-45-6789, can you look up the account?",
]

def bench(fn, samples, repeats):
    timings = []
    for _ in range(repeats):
        for s in samples:
            t = time.perf_counter()
            fn(s)
            timings.append(time.perf_counter() - t)
    return statistics.median(timings)

print("Benchmarking (this takes a moment for the classifier)...\n")

t_hand = bench(handrolled_check, SAMPLES, repeats=500)
t_gai  = bench(guardrails_ai_check, SAMPLES, repeats=25)
t_lg   = bench(llama_guard_check, SAMPLES, repeats=2)

results = [
    ("hand-rolled regex", t_hand, "none"),
    ("Guardrails-AI",     t_gai,  "guardrails-ai"),
    ("Llama Guard (1b)",  t_lg,   "ollama + 1B model"),
]

print(f"{'approach':<22s}{'median/call':>14s}{'vs hand-rolled':>18s}   dependency")
print("-" * 78)
for name, secs, dep in results:
    factor = secs / t_hand
    print(f"{name:<22s}{secs * 1000:>11.4f} ms{factor:>16,.0f}x   {dep}")


### Reading the benchmark

The numbers are stark, and the ratios matter more than the absolutes.

- **Hand-rolled** is in the microsecond range. At that cost you can run dozens
  of checks on every request and never notice.
- **Guardrails-AI** runs *the same regex* but pays framework overhead --
  validator dispatch, result objects, guard history tracking. It is still fast
  in human terms, but it is a large multiple of the raw check.
- **Llama Guard** is in the hundreds-of-milliseconds range: a full model
  inference. It is several orders of magnitude more expensive, and (as we just
  saw) it does not even catch this particular rule.

The honest summary: **for a rule you can write as a regex, wrapping it in a
framework buys you nothing and costs you latency.**

### 4. Where the framework genuinely wins

That was a case chosen to favour hand-rolling, and it is a very common case.
Now the opposite: a rule that **cannot be written down**.

> **The assistant must not give voting advice.**

There is no regex for this. There is no finite list of phrasings. The concept is
semantic. Below we pit a reasonable hand-rolled keyword filter against the
semantic approach on paraphrases that a real user might plausibly type.

### An unspecifiable rule: keyword filter vs semantic matching


In [ ]:
# A genuinely reasonable hand-rolled attempt -- not a strawman.
POLITICS_TERMS = [
    "vote", "voting", "election", "political party", "president",
    "democrat", "republican", "candidate", "ballot",
]

def handrolled_politics(text: str) -> bool:
    lowered = text.lower()
    return any(term in lowered for term in POLITICS_TERMS)

# The semantic approach: embed and compare, which is what NeMo's topical
# rail does internally. We use the same sentence-transformer family here.
from fastembed import TextEmbedding
import numpy as np

embedder = TextEmbedding()

EXAMPLES = [
    "who should I vote for",
    "what do you think about the election",
    "which political party is best",
    "is the president doing a good job",
]
example_vecs = np.array(list(embedder.embed(EXAMPLES)))

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def semantic_politics(text: str, threshold: float = 0.6) -> bool:
    vec = np.array(list(embedder.embed([text])))[0]
    return max(cosine(vec, e) for e in example_vecs) >= threshold

print("Embedder ready. Comparing the two approaches...\n")


### The comparison


In [ ]:
# (text, should_be_blocked)
politics_cases = [
    ("Who should I vote for in November?",                        True),
    ("Which political party is best?",                            True),
    # Paraphrases with NONE of the keywords:
    ("Honestly, which candidate deserves my ballot?",             True),
    ("Whose side should I take at the polls next month?",         True),
    ("Who do you reckon deserves to run the country?",            True),
    # Legitimate questions that CONTAIN the keywords:
    ("How do I vote on which feature you build next?",            False),
    ("Can I elect express shipping at checkout?",                 False),
    ("What is your return policy?",                               False),
]

print("=== Unspecifiable rule: 'no voting advice' ===\n")
print(f"{'text':<52s}{'want':>7s}{'regex':>9s}{'semantic':>11s}")
print("-" * 80)

hand_wrong = sem_wrong = 0
for text, want_blocked in politics_cases:
    h = handrolled_politics(text)
    s = semantic_politics(text)
    hand_wrong += (h != want_blocked)
    sem_wrong += (s != want_blocked)
    mark = lambda got: ("block" if got else "allow")
    flag_h = " " if h == want_blocked else "!"
    flag_s = " " if s == want_blocked else "!"
    print(f"{text[:50]:<52s}{('block' if want_blocked else 'allow'):>7s}"
          f"{mark(h):>8s}{flag_h}{mark(s):>10s}{flag_s}")

n = len(politics_cases)
print("-" * 80)
print(f"hand-rolled keyword filter : {n - hand_wrong}/{n} correct")
print(f"semantic matching          : {n - sem_wrong}/{n} correct")


### This is the case for the framework

Look at the two failure modes of the keyword filter:

- **False negatives.** `"Whose side should I take at the polls next month?"`
  contains no listed term. It sails straight through. You cannot fix this by
  adding words -- there is always another paraphrase.
- **False positives.** `"How do I vote on which feature you build next?"` is an
  ordinary product question that the filter blocks because the word "vote"
  appears. Adding more keywords makes this *worse*.

Those two errors pull in opposite directions. Every keyword you add to catch a
paraphrase increases the false-positive rate. There is no setting of the
keyword list that solves both, because the rule is about **meaning** and
keywords are about **strings**.

This is the real, non-negotiable case for a framework. When the rule is
semantic, you need a semantic mechanism, and you should not build one yourself.

### 5. The dimensions that don't show up in a benchmark

Latency is measurable, so it dominates discussions. These other factors usually
matter more in practice.

### Dependency weight

Adding a guardrail framework is not a small commitment. NeMo pulls in an
embedding runtime and downloads a model on first use. Guardrails-AI pulls in a
validation stack and, by default, a telemetry exporter that phones home. Both
pin versions of shared libraries that can conflict with the rest of your
application.

Your hand-rolled check imports `re`.

### Debuggability

This is the dimension that hurts most at 3 a.m.

- **Hand-rolled**: a guardrail misfires. You set a breakpoint in your own
  twenty-line function, look at the value, and see the answer. Total time:
  minutes.
- **Guardrails-AI**: a value came out different from what you expected. Was it a
  `FIX` policy silently rewriting it? Which validator in `use` ran first?
  Did an earlier fix change what a later validator saw? You are now reading
  library internals.
- **NeMo**: a rail did not fire. Was the similarity below threshold? Did the
  intent match a different flow? Is `embeddings_only` set the way you think? The
  failure is *silent and looks like success*.
- **Llama Guard**: the model said `safe` for something you consider unsafe.
  There is no explanation and nothing to step through. Your only recourse is
  prompt/taxonomy changes or a bigger model.

Debuggability degrades monotonically as you move up the stack -- and it degrades
fastest exactly where the failure is most dangerous.

### Failure mode when the guardrail itself breaks

| Approach | If it breaks | Direction |
|---|---|---|
| hand-rolled | raises, or visibly misbehaves | usually fails **closed** |
| Guardrails-AI | `FIX` reports success after modifying data | can hide problems |
| NeMo | rail silently does not match | fails **open** |
| Llama Guard | Ollama down -> exception, or you catch it and... ? | your choice, so choose |

**Failing open silently is the worst property a security control can have**, and
two of the four approaches have it by default.

### Make the "fails open" risk concrete


In [ ]:
print("=== What happens when the guardrail itself is misconfigured? ===\n")

# A hand-rolled check with a broken pattern: the error is loud and immediate.
def broken_handrolled(text):
    return bool(re.search(r"[unclosed", text))   # invalid regex

try:
    broken_handrolled("anything")
except re.error as e:
    print(f"hand-rolled, broken pattern -> raises immediately: {type(e).__name__}: {e}")
print("  You find out at import/first-call time. Loud. Fails closed.\n")

# A semantic rail with a threshold set too high: silently matches nothing.
print("semantic rail, threshold too high (0.95 instead of 0.6):")
for text in ["Who should I vote for in November?", "Which political party is best?"]:
    blocked = semantic_politics(text, threshold=0.95)
    print(f"  blocked={blocked!s:<5s}  {text}")
print("  No error. No warning. The rail simply never fires.")
print("  The application returns normal, helpful answers to everything.")
print("  This is what 'fails open' looks like: indistinguishable from working.")


That contrast is the single most important thing in this notebook.

A broken hand-rolled check **announces itself**. A broken semantic rail
**looks exactly like a working one**. If you adopt a framework, you take on the
obligation to write coverage tests that prove your rails still fire -- because
nothing else will tell you when they stop.

### 6. The decision procedure

Here is the rule to actually apply.

### Use hand-rolled (notebook `01` style) when...

- The rule is **exactly specifiable**: formats, ranges, allowlists, schemas,
  regex-able patterns, business invariants ("a refund over $100 needs a human").
- It runs on **every request** and latency matters.
- You need to **audit** it or show it to a compliance reviewer.
- It encodes **your domain**, which no library ships with anyway.

**This covers most guardrails in most applications.** PII patterns, output
schemas, tool-argument validation, rate and amount limits, allowlisted domains,
required fields. Every one of these is better as plain code.

### Use Guardrails-AI when...

- You have **many field-level checks** on structured output and want the
  per-field `on_fail` policy to be declarative rather than a wall of `if`s.
- You genuinely want **`REASK`** -- letting the model repair its own output is
  the one behaviour with no cheap hand-rolled equivalent.

### Use NeMo Guardrails when...

- The rule is **topical or conversational**: staying on-subject, refusing
  categories of discussion, multi-turn dialog policy.
- You need **semantic** matching over paraphrases, as demonstrated in section 4.

### Use Llama Guard when...

- You need **broad harm coverage** across categories you cannot enumerate, and
  you can afford a model inference per request.
- Use it as the **second layer**, after cheap checks have already filtered the
  obvious cases.

### The layered architecture

In practice you use several, ordered by cost:

```
1. hand-rolled deterministic checks   (microseconds)  -- most traffic exits here
2. framework validators / schema      (milliseconds)
3. semantic / topical rails           (milliseconds + embedding)
4. classifier model                   (hundreds of ms) -- only what survives
```

Cheapest first, so your median request never pays for the expensive layers. This
is the same principle as a cache hierarchy, and it is the right default.

### The layered pipeline, assembled and measured


In [ ]:
def layered_guard(text: str, use_classifier: bool = True) -> dict:
    """Cheapest checks first; escalate only if earlier layers pass."""
    trace = {"text": text, "layers_run": []}
    started = time.perf_counter()

    # Layer 1: deterministic, microseconds.
    trace["layers_run"].append("regex")
    ok, reason = handrolled_check(text)
    if not ok:
        trace.update(blocked=True, by="layer1:regex", reason=reason)
        trace["seconds"] = time.perf_counter() - started
        return trace

    # Layer 2: semantic topical rail, milliseconds.
    trace["layers_run"].append("semantic")
    if semantic_politics(text):
        trace.update(blocked=True, by="layer2:semantic", reason="off-topic (politics)")
        trace["seconds"] = time.perf_counter() - started
        return trace

    # Layer 3: classifier, hundreds of milliseconds.
    if use_classifier:
        trace["layers_run"].append("classifier")
        ok, reason = llama_guard_check(text)
        if not ok:
            trace.update(blocked=True, by="layer3:classifier", reason=reason)
            trace["seconds"] = time.perf_counter() - started
            return trace

    trace.update(blocked=False, by="", reason="")
    trace["seconds"] = time.perf_counter() - started
    return trace


print("=== Layered guardrail pipeline ===\n")
for text in [
    "My card is 4111 1111 1111 1111, please refund me.",   # layer 1
    "Who should I vote for in November?",                   # layer 2
    "How do I build a pipe bomb?",                          # layer 3
    "What is your return policy?",                          # passes all
]:
    r = layered_guard(text)
    verdict = f"BLOCKED by {r['by']}" if r["blocked"] else "ALLOWED"
    print(f"  {text[:48]:<50s}")
    print(f"    {verdict}   layers run: {'->'.join(r['layers_run'])}"
          f"   {r['seconds'] * 1000:.1f} ms")
    if r["reason"]:
        print(f"    reason: {r['reason']}")
    print()


Notice how much work the ordering saves. The credit-card message was rejected by
a regex in microseconds and never touched the embedder or the classifier. Only
the message that got past both cheap layers paid for a model inference.

If you inverted the order -- classifier first -- every one of those requests
would have cost hundreds of milliseconds for exactly the same verdicts.

### 7. Honest closing assessment

Some conclusions that are unfashionable but, on the evidence above, correct.

**The hand-rolled approach from notebook `01` is not a teaching toy.** It is the
right answer for narrow, well-understood checks, and narrow well-understood
checks are the majority of what a production system needs. Ten lines of regex
beat every framework in this module at detecting a leaked API key -- on latency,
on dependency weight, on debuggability, *and on accuracy*.

**Frameworks solve a specific problem, not a general one.** That problem is:
the rule is semantic and cannot be written down. When you have that problem, a
framework is close to indispensable, as section 4 demonstrated. When you do not
have that problem, you are paying its costs for none of its benefits.

**Adopting a framework transfers risk rather than removing it.** You trade "my
regex might have a bug I can find in five minutes" for "my rail might silently
stop matching and I will never know". The second is a *worse* failure mode. It
is worth accepting when the capability is worth it -- but it must be an accepted
trade, backed by coverage tests, not an assumption that the library has it
handled.

**The strongest systems layer several approaches** cheapest-first, and treat
every layer as fallible.

Finally: **a guardrail you have not tested is not a guardrail.** This applies
regardless of approach, but it applies most urgently to the framework-based
ones, precisely because they fail silently. Write the coverage test.

### Summary


In [ ]:
print("=== Notebook 08 Summary ===\n")
print(f"  measured cost per check:")
print(f"    hand-rolled regex : {t_hand * 1000:.4f} ms   (baseline)")
print(f"    Guardrails-AI     : {t_gai * 1000:.4f} ms   ({t_gai / t_hand:,.0f}x)")
print(f"    Llama Guard (1b)  : {t_lg * 1000:.1f} ms   ({t_lg / t_hand:,.0f}x)")
print()
print("  DECISION RULE: use the cheapest mechanism that can express the rule.")
print()
print("  hand-rolled  -> specifiable rules: formats, schemas, limits, patterns,")
print("                  business invariants. THIS IS MOST GUARDRAILS.")
print("  Guardrails-AI-> many field-level checks; or you specifically want REASK")
print("  NeMo         -> topical / conversational rules that need semantics")
print("  Llama Guard  -> broad harm coverage you cannot enumerate; second layer")
print()
print("  Framework advantage : semantic rules that cannot be written down")
print("  Framework cost      : latency, dependency weight, and silent fail-open")
print()
print("  Layer cheapest-first, and TEST that every rail still fires.")
print("\nModule 02 complete. Next: 03_caching_strategies.")
